In [1]:
# ============================================================
# 06 / Cell 1
# Detect city files and build a first city registry
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd

# ------------------------------------------------------------
# locate project root
# ------------------------------------------------------------
project_root = Path.cwd()
if not (project_root / "data_processed").exists():
    project_root = project_root.parent

output_dir = project_root / "outputs" / "pilot"
output_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)

# ------------------------------------------------------------
# city config
# ------------------------------------------------------------
CITY_CONFIG = [
    {"city": "Houston", "aliases": ["houston"]},
    {"city": "Phoenix", "aliases": ["phoenix"]},
    {"city": "Miami", "aliases": ["miami"]},
    {"city": "Las Vegas", "aliases": ["las_vegas", "lasvegas", "vegas"]},
]

# ------------------------------------------------------------
# helpers
# ------------------------------------------------------------
def list_candidate_files(root, aliases, suffixes=(".gpkg", ".geojson", ".csv")):
    out = []
    for p in root.rglob("*"):
        if not p.is_file():
            continue
        if p.suffix.lower() not in suffixes:
            continue
        low = str(p).lower()
        if any(a in low for a in aliases):
            out.append(p)
    return sorted(out)

def score_main_gpkg(p: Path):
    low = p.name.lower()
    score = 0
    if p.suffix.lower() == ".gpkg":
        score += 10
    if "master" in low:
        score += 8
    if "fixed" in low:
        score += 6
    if "compare" in low:
        score += 5
    if "lst" in low:
        score += 4
    if "hi" in low:
        score += 4
    if "heat" in low:
        score += 2
    if "demo" in low:
        score += 1
    if "boundary" in low:
        score -= 8
    if "lookup" in low:
        score -= 8
    return score

def score_boundary_file(p: Path):
    low = p.name.lower()
    score = 0
    if "boundary" in low:
        score += 10
    if "heat_ready" in low:
        score += 4
    if p.suffix.lower() in [".geojson", ".gpkg"]:
        score += 3
    if "tract" in low:
        score -= 6
    if "lookup" in low:
        score -= 6
    return score

def pick_first_column(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

def inspect_gpkg(gpkg_path: Path):
    try:
        gdf = gpd.read_file(gpkg_path)
    except Exception as e:
        return {
            "rows": np.nan,
            "crs": None,
            "lst_col": None,
            "hi_col": None,
            "pop_col": None,
            "geoid_col": None,
            "error": str(e),
            "columns": None,
        }

    cols = list(gdf.columns)

    lst_col = pick_first_column(cols, [
        "lst_c", "lst_value", "lst_summer_median_c", "median_lst_c",
        "median_lst", "lst_median_c", "lst_mean_c", "lst_sum_mean_c"
    ])

    hi_col = pick_first_column(cols, [
        "hi_c", "hi_value", "hi_summer_median_c", "median_hi_c",
        "hi_median_c", "median_heat_index_c", "heat_index_median_c",
        "hi_mean_c", "hi_sum_mean_c"
    ])

    pop_col = pick_first_column(cols, [
        "total_population", "population", "tot_pop", "pop",
        "B01003_001E", "POPULATION", "Total_Population"
    ])

    geoid_col = pick_first_column(cols, [
        "GEOID", "geoid", "tract_geoid", "tract_id", "GEOIDFQ", "TRACTCE"
    ])

    return {
        "rows": len(gdf),
        "crs": str(gdf.crs),
        "lst_col": lst_col,
        "hi_col": hi_col,
        "pop_col": pop_col,
        "geoid_col": geoid_col,
        "error": None,
        "columns": cols[:40],  # preview only
    }

# ------------------------------------------------------------
# search and detect
# ------------------------------------------------------------
records = []

for cfg in CITY_CONFIG:
    city = cfg["city"]
    aliases = cfg["aliases"]

    dp_files = list_candidate_files(project_root / "data_processed", aliases, suffixes=(".gpkg", ".geojson", ".csv"))
    out_files = list_candidate_files(project_root / "outputs", aliases, suffixes=(".gpkg", ".geojson", ".csv"))

    gpkg_candidates = [p for p in dp_files if p.suffix.lower() == ".gpkg"]
    boundary_candidates = [p for p in (dp_files + out_files) if p.suffix.lower() in [".gpkg", ".geojson"] and "boundary" in p.name.lower()]

    main_gpkg = None
    if gpkg_candidates:
        main_gpkg = sorted(gpkg_candidates, key=score_main_gpkg, reverse=True)[0]

    boundary_file = None
    if boundary_candidates:
        boundary_file = sorted(boundary_candidates, key=score_boundary_file, reverse=True)[0]

    inspect = {
        "rows": np.nan,
        "crs": None,
        "lst_col": None,
        "hi_col": None,
        "pop_col": None,
        "geoid_col": None,
        "error": None,
        "columns": None,
    }
    if main_gpkg is not None:
        inspect = inspect_gpkg(main_gpkg)

    records.append({
        "city": city,
        "aliases": ",".join(aliases),
        "main_gpkg": str(main_gpkg) if main_gpkg is not None else None,
        "boundary_file": str(boundary_file) if boundary_file is not None else None,
        "rows": inspect["rows"],
        "crs": inspect["crs"],
        "lst_col": inspect["lst_col"],
        "hi_col": inspect["hi_col"],
        "pop_col": inspect["pop_col"],
        "geoid_col": inspect["geoid_col"],
        "read_error": inspect["error"],
        "n_data_processed_matches": len(dp_files),
        "n_outputs_matches": len(out_files),
        "column_preview": str(inspect["columns"]),
    })

registry_df = pd.DataFrame(records)

display(registry_df)

registry_out = output_dir / "rq1_city_registry_detected.csv"
registry_df.to_csv(registry_out, index=False)

print("\nSaved:", registry_out)
print(registry_out.exists())

print("\nQuick check:")
for _, row in registry_df.iterrows():
    print("-" * 70)
    print("City:", row["city"])
    print("main_gpkg:", row["main_gpkg"])
    print("boundary_file:", row["boundary_file"])
    print("lst_col:", row["lst_col"], "| hi_col:", row["hi_col"], "| pop_col:", row["pop_col"], "| geoid_col:", row["geoid_col"])
    print("rows:", row["rows"], "| crs:", row["crs"])
    if pd.notna(row["read_error"]) and row["read_error"] not in [None, "", "nan"]:
        print("ERROR:", row["read_error"])

Project root: /Users/yufeizhou/Desktop/heat-exposure-compare


,city,aliases,main_gpkg,boundary_file,rows,crs,lst_col,hi_col,pop_col,geoid_col,read_error,n_data_processed_matches,n_outputs_matches,column_preview
0,Houston,houston,/Users/yufeizhou/Desktop/heat-exposure-compare...,/Users/yufeizhou/Desktop/heat-exposure-compare...,654.0,EPSG:32615,lst_c,hi_c,total_population,GEOID,None,14,7,"['STATEFP', 'COUNTYFP', 'TRACTCE', 'GEOID', 'N..."
1,Phoenix,phoenix,/Users/yufeizhou/Desktop/heat-exposure-compare...,/Users/yufeizhou/Desktop/heat-exposure-compare...,373.0,EPSG:32612,lst_c,hi_c,total_population,GEOID,None,14,7,"['STATEFP', 'COUNTYFP', 'TRACTCE', 'GEOID', 'N..."
2,Miami,miami,None,None,NaN,None,None,None,None,None,None,0,0,None
3,Las Vegas,"las_vegas,lasvegas,vegas",None,None,NaN,None,None,None,None,None,None,0,0,None



Saved: /Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/rq1_city_registry_detected.csv
True

Quick check:
----------------------------------------------------------------------
City: Houston
main_gpkg: /Users/yufeizhou/Desktop/heat-exposure-compare/data_processed/houston/houston_master_with_lst_hi_fixed.gpkg
boundary_file: /Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/heat_ready/houston/houston_boundary_heat_ready.geojson
lst_col: lst_c | hi_col: hi_c | pop_col: total_population | geoid_col: GEOID
rows: 654.0 | crs: EPSG:32615
----------------------------------------------------------------------
City: Phoenix
main_gpkg: /Users/yufeizhou/Desktop/heat-exposure-compare/data_processed/phoenix/phoenix_master_with_lst_hi_fixed.gpkg
boundary_file: /Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/heat_ready/phoenix/phoenix_boundary_heat_ready.geojson
lst_col: lst_c | hi_col: hi_c | pop_col: total_population | geoid_col: GEOID
rows: 373.0 | crs: EPSG:326

In [2]:
# ============================================================
# 06 / Cell 2
# Build first multicity summary table
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd

project_root = Path.cwd()
if not (project_root / "data_processed").exists():
    project_root = project_root.parent

output_dir = project_root / "outputs" / "pilot"
registry_path = output_dir / "rq1_city_registry_detected.csv"

registry_df = pd.read_csv(registry_path)

summary_rows = []

for _, row in registry_df.iterrows():
    city = row["city"]
    gpkg = row["main_gpkg"]
    lst_col = row["lst_col"]
    hi_col = row["hi_col"]
    pop_col = row["pop_col"]
    geoid_col = row["geoid_col"]

    if pd.isna(gpkg) or pd.isna(lst_col) or pd.isna(hi_col):
        summary_rows.append({
            "city": city,
            "status": "missing_required_inputs",
            "tract_rows": np.nan,
            "valid_both_n": np.nan,
            "missing_hi_n": np.nan,
            "median_lst_c": np.nan,
            "median_hi_c": np.nan,
            "lst_hotspot_threshold_top20_c": np.nan,
            "hi_hotspot_threshold_top20_c": np.nan,
            "lst_hotspot_n": np.nan,
            "hi_hotspot_n": np.nan,
            "overlap_n": np.nan,
            "lst_only_n": np.nan,
            "hi_only_n": np.nan,
            "jaccard_index": np.nan,
            "overlap_share_pct": np.nan,
            "lst_only_share_pct": np.nan,
            "hi_only_share_pct": np.nan,
            "population_overlap_share_pct": np.nan,
            "geoid_col": geoid_col,
            "pop_col": pop_col,
        })
        continue

    try:
        gdf = gpd.read_file(gpkg).copy()
    except Exception as e:
        summary_rows.append({
            "city": city,
            "status": f"read_failed: {e}",
            "tract_rows": np.nan,
            "valid_both_n": np.nan,
            "missing_hi_n": np.nan,
            "median_lst_c": np.nan,
            "median_hi_c": np.nan,
            "lst_hotspot_threshold_top20_c": np.nan,
            "hi_hotspot_threshold_top20_c": np.nan,
            "lst_hotspot_n": np.nan,
            "hi_hotspot_n": np.nan,
            "overlap_n": np.nan,
            "lst_only_n": np.nan,
            "hi_only_n": np.nan,
            "jaccard_index": np.nan,
            "overlap_share_pct": np.nan,
            "lst_only_share_pct": np.nan,
            "hi_only_share_pct": np.nan,
            "population_overlap_share_pct": np.nan,
            "geoid_col": geoid_col,
            "pop_col": pop_col,
        })
        continue

    gdf = gdf.loc[gdf.geometry.notna()].copy()
    gdf["lst_c_use"] = pd.to_numeric(gdf[lst_col], errors="coerce")
    gdf["hi_c_use"] = pd.to_numeric(gdf[hi_col], errors="coerce")

    if pop_col in gdf.columns:
        gdf["pop_use"] = pd.to_numeric(gdf[pop_col], errors="coerce")
    else:
        gdf["pop_use"] = np.nan

    tract_rows = len(gdf)
    valid_both = gdf["lst_c_use"].notna() & gdf["hi_c_use"].notna()
    valid_both_n = int(valid_both.sum())
    missing_hi_n = int(gdf["hi_c_use"].isna().sum())

    lst_thr = gdf.loc[gdf["lst_c_use"].notna(), "lst_c_use"].quantile(0.80)
    hi_thr = gdf.loc[gdf["hi_c_use"].notna(), "hi_c_use"].quantile(0.80)

    gdf["lst_hotspot"] = gdf["lst_c_use"] >= lst_thr
    gdf["hi_hotspot"] = gdf["hi_c_use"] >= hi_thr

    gdf["compare_group"] = np.select(
        [
            valid_both & gdf["lst_hotspot"] & gdf["hi_hotspot"],
            valid_both & gdf["lst_hotspot"] & (~gdf["hi_hotspot"]),
            valid_both & (~gdf["lst_hotspot"]) & gdf["hi_hotspot"],
            valid_both & (~gdf["lst_hotspot"]) & (~gdf["hi_hotspot"]),
        ],
        [
            "Overlap hotspot",
            "LST-only hotspot",
            "HI-only hotspot",
            "Neither",
        ],
        default="Missing"
    )

    overlap_n = int((gdf["compare_group"] == "Overlap hotspot").sum())
    lst_only_n = int((gdf["compare_group"] == "LST-only hotspot").sum())
    hi_only_n = int((gdf["compare_group"] == "HI-only hotspot").sum())

    union_n = overlap_n + lst_only_n + hi_only_n
    jaccard = overlap_n / union_n if union_n > 0 else np.nan

    overlap_share_pct = overlap_n / valid_both_n * 100 if valid_both_n > 0 else np.nan
    lst_only_share_pct = lst_only_n / valid_both_n * 100 if valid_both_n > 0 else np.nan
    hi_only_share_pct = hi_only_n / valid_both_n * 100 if valid_both_n > 0 else np.nan

    pop_valid = gdf.loc[valid_both & gdf["pop_use"].notna()].copy()
    pop_total = pop_valid["pop_use"].sum()
    pop_overlap = pop_valid.loc[pop_valid["compare_group"] == "Overlap hotspot", "pop_use"].sum()
    pop_overlap_share_pct = pop_overlap / pop_total * 100 if pop_total > 0 else np.nan

    summary_rows.append({
        "city": city,
        "status": "ok",
        "tract_rows": tract_rows,
        "valid_both_n": valid_both_n,
        "missing_hi_n": missing_hi_n,
        "median_lst_c": gdf["lst_c_use"].median(),
        "median_hi_c": gdf["hi_c_use"].median(),
        "lst_hotspot_threshold_top20_c": lst_thr,
        "hi_hotspot_threshold_top20_c": hi_thr,
        "lst_hotspot_n": int(gdf["lst_hotspot"].sum()),
        "hi_hotspot_n": int(gdf["hi_hotspot"].sum()),
        "overlap_n": overlap_n,
        "lst_only_n": lst_only_n,
        "hi_only_n": hi_only_n,
        "jaccard_index": jaccard,
        "overlap_share_pct": overlap_share_pct,
        "lst_only_share_pct": lst_only_share_pct,
        "hi_only_share_pct": hi_only_share_pct,
        "population_overlap_share_pct": pop_overlap_share_pct,
        "geoid_col": geoid_col,
        "pop_col": pop_col,
    })

summary_df = pd.DataFrame(summary_rows)

display(summary_df.round(3))

summary_out = output_dir / "rq1_multicity_summary_v1.csv"
summary_df.to_csv(summary_out, index=False)

print("\nSaved:", summary_out)
print(summary_out.exists())

,city,status,tract_rows,valid_both_n,missing_hi_n,median_lst_c,median_hi_c,lst_hotspot_threshold_top20_c,hi_hotspot_threshold_top20_c,lst_hotspot_n,...,overlap_n,lst_only_n,hi_only_n,jaccard_index,overlap_share_pct,lst_only_share_pct,hi_only_share_pct,population_overlap_share_pct,geoid_col,pop_col
0,Houston,ok,654.0,549.0,105.0,48.531,34.182,50.315,34.198,131.0,...,38.0,45.0,108.0,0.199,6.922,8.197,19.672,6.188,GEOID,total_population
1,Phoenix,ok,373.0,332.0,41.0,57.217,33.308,58.607,33.694,75.0,...,29.0,37.0,44.0,0.264,8.735,11.145,13.253,9.344,GEOID,total_population
2,Miami,missing_required_inputs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Las Vegas,missing_required_inputs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Saved: /Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/rq1_multicity_summary_v1.csv
True


In [3]:
# ============================================================
# 06 / Cell 3
# Repo-wide inventory scan for all target cities
# ============================================================

from pathlib import Path
import pandas as pd

project_root = Path.cwd()
if not (project_root / "data_processed").exists():
    project_root = project_root.parent

output_dir = project_root / "outputs" / "pilot"
output_dir.mkdir(parents=True, exist_ok=True)

CITY_SCAN = [
    {"city": "Houston", "aliases": ["houston"]},
    {"city": "Phoenix", "aliases": ["phoenix"]},
    {"city": "Miami", "aliases": ["miami"]},
    {"city": "Las Vegas", "aliases": ["las_vegas", "lasvegas", "vegas"]},
]

SEARCH_DIRS = [
    project_root / "data_processed",
    project_root / "data_raw",
    project_root / "outputs",
    project_root / "docs",
    project_root / "notebooks",
    project_root / "archive_old",
]

VALID_SUFFIXES = {
    ".gpkg", ".geojson", ".csv", ".parquet", ".json",
    ".tif", ".tiff", ".txt", ".md", ".ipynb", ".html"
}

rows = []

for cfg in CITY_SCAN:
    city = cfg["city"]
    aliases = cfg["aliases"]

    for base in SEARCH_DIRS:
        if not base.exists():
            continue

        for p in base.rglob("*"):
            if not p.is_file():
                continue
            if p.suffix.lower() not in VALID_SUFFIXES:
                continue

            low = str(p).lower()
            if any(a in low for a in aliases):
                rows.append({
                    "city": city,
                    "base_dir": str(base.relative_to(project_root)),
                    "relative_path": str(p.relative_to(project_root)),
                    "suffix": p.suffix.lower(),
                    "filename": p.name
                })

inventory_df = pd.DataFrame(rows).sort_values(["city", "base_dir", "relative_path"]).reset_index(drop=True)

display(inventory_df)

inventory_out = output_dir / "rq1_repo_city_inventory.csv"
inventory_df.to_csv(inventory_out, index=False)

print("\nSaved:", inventory_out)
print(inventory_out.exists())

print("\nCounts by city:")
if len(inventory_df) > 0:
    print(inventory_df.groupby("city").size())
else:
    print("No matching files found.")

,city,base_dir,relative_path,suffix,filename
0,Houston,data_processed,data_processed/houston/houston_city_boundary.gpkg,.gpkg,houston_city_boundary.gpkg
1,Houston,data_processed,data_processed/houston/houston_city_tracts.gpkg,.gpkg,houston_city_tracts.gpkg
2,Houston,data_processed,data_processed/houston/houston_master_demo.gpkg,.gpkg,houston_master_demo.gpkg
3,Houston,data_processed,data_processed/houston/houston_master_demo_cle...,.gpkg,houston_master_demo_clean.gpkg
4,Houston,data_processed,data_processed/houston/houston_master_demo_pov...,.gpkg,houston_master_demo_poverty.gpkg
5,Houston,data_processed,data_processed/houston/houston_master_gistar.gpkg,.gpkg,houston_master_gistar.gpkg
6,Houston,data_processed,data_processed/houston/houston_master_lst_hi_h...,.gpkg,houston_master_lst_hi_hotcompare.gpkg
7,Houston,data_processed,data_processed/houston/houston_master_with_lst...,.gpkg,houston_master_with_lst.gpkg
8,Houston,data_processed,data_processed/houston/houston_master_with_lst...,.gpkg,houston_master_with_lst_hi.gpkg
9,Houston,data_processed,data_processed/houston/houston_master_with_lst...,.gpkg,houston_master_with_lst_hi_fixed.gpkg



Saved: /Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/rq1_repo_city_inventory.csv
True

Counts by city:
city
Houston    21
Phoenix    21
dtype: int64


In [4]:
# ============================================================
# 06 / Cell 4
# Build reusable city registry template
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd()
if not (project_root / "data_processed").exists():
    project_root = project_root.parent

output_dir = project_root / "outputs" / "pilot"

detected = pd.read_csv(output_dir / "rq1_city_registry_detected.csv")

target_cities = ["Houston", "Phoenix", "Miami", "Las Vegas"]

template_rows = []

for city in target_cities:
    sub = detected.loc[detected["city"] == city]

    if len(sub) == 1 and pd.notna(sub.iloc[0]["main_gpkg"]):
        r = sub.iloc[0]
        template_rows.append({
            "city": city,
            "is_active": True,
            "status": "ready" if pd.notna(r["lst_col"]) and pd.notna(r["hi_col"]) else "needs_column_fix",
            "main_gpkg": r["main_gpkg"],
            "boundary_file": r["boundary_file"],
            "lst_col": r["lst_col"],
            "hi_col": r["hi_col"],
            "pop_col": r["pop_col"],
            "geoid_col": r["geoid_col"],
            "notes": "Auto-filled from detected registry"
        })
    else:
        template_rows.append({
            "city": city,
            "is_active": False,
            "status": "missing_inputs",
            "main_gpkg": None,
            "boundary_file": None,
            "lst_col": None,
            "hi_col": None,
            "pop_col": None,
            "geoid_col": None,
            "notes": "Need merged tract-level gpkg + boundary + columns before batch run"
        })

registry_template = pd.DataFrame(template_rows)

display(registry_template)

registry_template_out = output_dir / "rq1_city_registry_template.csv"
registry_template.to_csv(registry_template_out, index=False)

print("\nSaved:", registry_template_out)
print(registry_template_out.exists())

,city,is_active,status,main_gpkg,boundary_file,lst_col,hi_col,pop_col,geoid_col,notes
0,Houston,True,ready,/Users/yufeizhou/Desktop/heat-exposure-compare...,/Users/yufeizhou/Desktop/heat-exposure-compare...,lst_c,hi_c,total_population,GEOID,Auto-filled from detected registry
1,Phoenix,True,ready,/Users/yufeizhou/Desktop/heat-exposure-compare...,/Users/yufeizhou/Desktop/heat-exposure-compare...,lst_c,hi_c,total_population,GEOID,Auto-filled from detected registry
2,Miami,False,missing_inputs,None,None,None,None,None,None,Need merged tract-level gpkg + boundary + colu...
3,Las Vegas,False,missing_inputs,None,None,None,None,None,None,Need merged tract-level gpkg + boundary + colu...



Saved: /Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/rq1_city_registry_template.csv
True


In [6]:
# ============================================================
# 06 / Cell 5  (FULL REPLACEMENT)
# Export paper-ready two-city baseline tables
# no tabulate dependency
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd()
if not (project_root / "data_processed").exists():
    project_root = project_root.parent

output_dir = project_root / "outputs" / "pilot"

summary_df = pd.read_csv(output_dir / "rq1_multicity_summary_v1.csv")

twocity = summary_df.loc[summary_df["status"] == "ok"].copy()

table1 = twocity[[
    "city",
    "tract_rows",
    "valid_both_n",
    "missing_hi_n",
    "median_lst_c",
    "median_hi_c",
    "lst_hotspot_threshold_top20_c",
    "hi_hotspot_threshold_top20_c",
    "lst_hotspot_n",
    "hi_hotspot_n",
    "overlap_n",
    "lst_only_n",
    "hi_only_n",
    "jaccard_index",
    "overlap_share_pct",
    "lst_only_share_pct",
    "hi_only_share_pct",
    "population_overlap_share_pct",
]].copy()

table1 = table1.rename(columns={
    "tract_rows": "tract_count",
    "valid_both_n": "valid_both_count",
    "missing_hi_n": "missing_hi_count",
    "lst_hotspot_threshold_top20_c": "lst_hotspot_threshold_c",
    "hi_hotspot_threshold_top20_c": "hi_hotspot_threshold_c",
    "lst_hotspot_n": "lst_hotspot_count",
    "hi_hotspot_n": "hi_hotspot_count",
    "overlap_n": "overlap_hotspot_count",
    "lst_only_n": "lst_only_hotspot_count",
    "hi_only_n": "hi_only_hotspot_count",
})

# round for display/export
table1_round = table1.copy()
num_cols = [c for c in table1_round.columns if c != "city"]
table1_round[num_cols] = table1_round[num_cols].round(3)

display(table1_round)

table1_csv_out = output_dir / "rq1_table1_twocity_baseline.csv"
table1_round.to_csv(table1_csv_out, index=False)

# ------------------------------------------------------------
# manual markdown export (no tabulate needed)
# ------------------------------------------------------------
def df_to_simple_markdown(df: pd.DataFrame) -> str:
    cols = list(df.columns)
    header = "| " + " | ".join(cols) + " |"
    sep = "| " + " | ".join(["---"] * len(cols)) + " |"

    lines = [header, sep]
    for _, row in df.iterrows():
        vals = []
        for c in cols:
            v = row[c]
            if pd.isna(v):
                vals.append("")
            else:
                vals.append(str(v))
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)

table1_md = df_to_simple_markdown(table1_round)
table1_md_out = output_dir / "rq1_table1_twocity_baseline.md"
table1_md_out.write_text(table1_md, encoding="utf-8")

print("\nSaved:")
print(table1_csv_out, "->", table1_csv_out.exists())
print(table1_md_out, "->", table1_md_out.exists())

print("\n----- Table 1 markdown preview -----\n")
print(table1_md)

# ------------------------------------------------------------
# pending city checklist
# ------------------------------------------------------------
pending = summary_df.loc[summary_df["status"] != "ok", ["city", "status"]].copy()
pending["next_step"] = "Need tract-level merged gpkg with LST + HI + population columns"

display(pending)

pending_out = output_dir / "rq1_pending_city_checklist.csv"
pending.to_csv(pending_out, index=False)

print("\nSaved pending checklist:")
print(pending_out, "->", pending_out.exists())

# ------------------------------------------------------------
# concise interpretation note
# ------------------------------------------------------------
note_lines = []
note_lines.append("# RQ1 two-city baseline note")
note_lines.append("")
note_lines.append("Current batch-ready cities: Houston and Phoenix.")
note_lines.append("")
for _, row in table1_round.iterrows():
    note_lines.append(
        f"- {row['city']}: Jaccard={row['jaccard_index']}, "
        f"overlap share={row['overlap_share_pct']}%, "
        f"LST-only share={row['lst_only_share_pct']}%, "
        f"HI-only share={row['hi_only_share_pct']}%."
    )
note_lines.append("")
note_lines.append("Miami and Las Vegas are not yet batch-ready because merged tract-level inputs are missing.")

note_out = output_dir / "rq1_twocity_baseline_note.md"
note_out.write_text("\n".join(note_lines), encoding="utf-8")

print(note_out, "->", note_out.exists())

,city,tract_count,valid_both_count,missing_hi_count,median_lst_c,median_hi_c,lst_hotspot_threshold_c,hi_hotspot_threshold_c,lst_hotspot_count,hi_hotspot_count,overlap_hotspot_count,lst_only_hotspot_count,hi_only_hotspot_count,jaccard_index,overlap_share_pct,lst_only_share_pct,hi_only_share_pct,population_overlap_share_pct
0,Houston,654.0,549.0,105.0,48.531,34.182,50.315,34.198,131.0,146.0,38.0,45.0,108.0,0.199,6.922,8.197,19.672,6.188
1,Phoenix,373.0,332.0,41.0,57.217,33.308,58.607,33.694,75.0,73.0,29.0,37.0,44.0,0.264,8.735,11.145,13.253,9.344



Saved:
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/rq1_table1_twocity_baseline.csv -> True
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/rq1_table1_twocity_baseline.md -> True

----- Table 1 markdown preview -----

| city | tract_count | valid_both_count | missing_hi_count | median_lst_c | median_hi_c | lst_hotspot_threshold_c | hi_hotspot_threshold_c | lst_hotspot_count | hi_hotspot_count | overlap_hotspot_count | lst_only_hotspot_count | hi_only_hotspot_count | jaccard_index | overlap_share_pct | lst_only_share_pct | hi_only_share_pct | population_overlap_share_pct |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Houston | 654.0 | 549.0 | 105.0 | 48.531 | 34.182 | 50.315 | 34.198 | 131.0 | 146.0 | 38.0 | 45.0 | 108.0 | 0.199 | 6.922 | 8.197 | 19.672 | 6.188 |
| Phoenix | 373.0 | 332.0 | 41.0 | 57.217 | 33.308 | 58.607 | 33.694 | 75.0 | 73.0 | 29.0 | 37.0 | 44.0 | 0.264 | 8.735 | 11.145 |

,city,status,next_step
2,Miami,missing_required_inputs,Need tract-level merged gpkg with LST + HI + p...
3,Las Vegas,missing_required_inputs,Need tract-level merged gpkg with LST + HI + p...



Saved pending checklist:
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/rq1_pending_city_checklist.csv -> True
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/rq1_twocity_baseline_note.md -> True
